In [31]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle')
INPUT_DIR = PROJECT_ROOT / 'input'
WORKING_DIR = PROJECT_ROOT / 'working'

CONFIGS_DIR = PROJECT_ROOT / 'configs'

DATA_DIR = INPUT_DIR / 'datasets' / 'ahmedmohamed365'
VIDEOS_DIR = DATA_DIR / 'volleyball' / 'volleyball_' / 'videos'
DEFAULT_ANNOTATIONS_DIR = DATA_DIR / 'volleyball' / 'volleyball_tracking_annotation' /'volleyball_tracking_annotation'

OUTPUT_DIR = WORKING_DIR / 'outputs'

PERSON_ANNOTATIONS_DIR = OUTPUT_DIR / 'persons_annotations'
IMAGE_ANNOTATIONS_DIR = OUTPUT_DIR / 'images_annotations'

EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'


# print(PROJECT_ROOT)
# print(CONFIGS_DIR)
# print(DATA_DIR)
# print(OUTPUT_DIR)



In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [3]:
def get_videos_dirs():
    train_dirs = ["1", "3", "6", "7", "10", "13", "15", "16", "18", "22", "23", "31", "32", "36", "38", "39", "40",
                 "41", "42", "48", "50", "52", "53", "54"]
    train_dirs.sort()

    val_dirs = ["0", "2", "8", "12", "17", "19", "24", "26", "27", "28", "30", "33", "46", "49", "51"]
    val_dirs.sort()

    test_dirs = ['4', '5', '9', '11', '14', '20', '21', '25', '29', '34', '35', '37', '43', '44', '45', '47']
    test_dirs.sort()

    return train_dirs, val_dirs, test_dirs

### Group and Person Category

In [4]:
from logging import root
import os
import pickle

def prep_categories():
    group_categories = {
        'l-pass': 0,
        'r-pass': 1,
        'l-spike': 2,
        'r_spike': 3,
        'l_set': 4,
        'r_set': 5,
        'l_winpoint': 6,
        'r_winpoint': 7
    }

    person_categories = {
        'standing': 0,
        'setting': 1,
        'waiting': 2,
        'moving': 3,
        'falling': 4,
        'spiking': 5,
        'jumping': 6,
        'digging': 7,
        'blocking': 8
    }
    
    return group_categories, person_categories


### Box Info (Comparable)

In [5]:
# Box info for every single player
class BoxInfo:
    def __init__(self, line):
        words = line.split()

        self.category = words.pop()
        words = [int(word) for word in words]

        player_id, x1, y1, x2, y2, frame_id, lost, grouping, generated = words
        self.player_id = player_id
        self.box = [x1, y1, x2, y2]
        self.frame_id = frame_id
        self.lost = lost
        self.grouping = grouping
        self.generated = generated

    def get_box_info(self):
        return {'frame_id': self.frame_id,
                'box': self.box,
                'category': self.category
                }

### Tracking annotation for every single player within 20 frame per clip for the 12 players  (Comparable)

In [6]:
def get_players_boxes(tracking_annot_path):
    # load tracking annotations for one clip
    with open(tracking_annot_path, 'r') as file:
        player_boxes = {idx:[] for idx in range(12)}

        for line in file:
            box_info = BoxInfo(line)
            player_id, box, lost, category = box_info.player_id, box_info.box, box_info.lost, box_info.category
            box_info_dct = box_info.get_box_info()

            # if number of players is more than 12 by mistake stop on player number 12 and ignore others
            if box_info.player_id > 11:
                continue

            player_boxes[box_info.player_id].append(box_info_dct)

        frame_boxes_dct = {}
        for player_id, boxes_info in player_boxes.items():
            # for baseline 3, I need just 4 frames before and 4 after the target [5:13] from 6 to 14 ignoring zero indexing
            # boxes_info = boxes_info[5:-6]
            if len(boxes_info) == 0:
                continue
            
            target_boxes_info = boxes_info[9]

            player_box = {
                        'player_id': player_id,
                        'box': target_boxes_info['box'],
                        'category': target_boxes_info['category']
            }

            if target_boxes_info['frame_id'] not in frame_boxes_dct:
                frame_boxes_dct[target_boxes_info['frame_id']] = []

            frame_boxes_dct[target_boxes_info['frame_id']].append(player_box)
        
        frame_boxes_lst_dct = {}
        for frame_id, boxes_lst in frame_boxes_dct.items():
            player_id = []
            box = []
            category = []

            for boxes in boxes_lst:
                player_id.append(boxes['player_id'])
                box.append(boxes['box'])
                category.append(boxes['category'])

            frame_boxes_lst_dct[frame_id] = [player_id, box, category]
            # print(frame_boxes_lst_dct[frame_id])

        '''
        It returns 9 frames per clip, each frame has 12 player boxes.
        RETURN: --> { 'frame_id': [[player0, player2, ......,player11]
                                   [box0, box1, ......., box11]
                                   [cat0, cat1, ......, cat11]]
                                   }
        '''
        # print(frame_boxes_lst_dct)
        return frame_boxes_lst_dct

In [7]:
# tracking_annot_path = '/kaggle/input/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation'

def load_tracking_annots(dirs_list, tracking_annot_path: str):
    print(f'Loading tracking annotations for {len(dirs_list)} videos from {tracking_annot_path}')
    clip_track_annots = []
    person_activity_encode = prep_categories()[1]

    for vid in dirs_list:
        if not os.path.exists(os.path.join(tracking_annot_path, vid)):
            continue

        tracking_annots_vid_dir = os.listdir(os.path.join(tracking_annot_path, vid))
        tracking_annots_vid_dir.sort()
        # print(clips_dir)
        # print(tracking_annots_dir)

        for clip in tracking_annots_vid_dir:
            if not os.path.isdir(os.path.join(tracking_annot_path, vid, clip)):
                continue

            players_boxes = get_players_boxes(os.path.join(tracking_annot_path, vid, clip, f'{clip}.txt'))

            for frame_id, boxes in players_boxes.items():
                category = []
                for cat in boxes[2]:
                    category.append(person_activity_encode[cat])

                clip_track_annots_players_dct = {
                    'video': vid,
                    'clip': clip,
                    'frame_id': frame_id,
                    'boxes': boxes[1],
                    'category': category
                }
                # print(clip_track_annots_players_dct)
                clip_track_annots.append(clip_track_annots_players_dct)
            # print(clip_track_annots)
    return clip_track_annots

In [8]:
import pathlib

# root_path = 'kaggle/input'

training_output_path = "/kaggle/working/training-outputs"
if not os.path.exists(training_output_path):
    os.makedirs(training_output_path)

def prepare_persons_annotations():
    # get train, val and test folders in a sorted way

    annotations_save_path = PERSON_ANNOTATIONS_DIR
    if not os.path.exists(annotations_save_path):
        os.makedirs(annotations_save_path)

    tracking_annots_path = DEFAULT_ANNOTATIONS_DIR
    train, val, test = get_videos_dirs()

    train_annots = load_tracking_annots(train, tracking_annots_path)
    print(len(train_annots))
    # train_crops = get_cropped_images(videos_path, train_annots)

    # print('the length of annot : ' + str(len(train_annots)))
    # print(type(train_annots[0]))
    # print((train_annots[0]))
    # print(type(train_annots[0]['box']))

    val_annots = load_tracking_annots(val, tracking_annots_path)
    print(len(val_annots))
    # val_crops = get_cropped_images(videos_path, val_annots)

    test_annots = load_tracking_annots(test, tracking_annots_path)
    # test_crops = get_cropped_images(videos_path, test_annots)

    with open(annotations_save_path / 'train_players_crops.pickle', 'wb') as tr_file:
        pickle.dump(train_annots, tr_file, pickle.HIGHEST_PROTOCOL)

    bytes_size = os.path.getsize(os.path.join(annotations_save_path, 'train_players_crops.pickle'))
    mb_size = bytes_size / (1024 * 1024)

    print(f"File size: {mb_size:.2f} MB")

    with open(annotations_save_path / 'val_players_crops.pickle', 'wb') as vl_file:
        pickle.dump(val_annots, vl_file, pickle.HIGHEST_PROTOCOL)
    #
    with open(annotations_save_path / 'test_players_crops.pickle', 'wb') as ts_file:
        pickle.dump(test_annots, ts_file, pickle.HIGHEST_PROTOCOL)

it took 15 minutes to prepare the train/val/test data pickle files

In [9]:
prepare_persons_annotations()

Loading tracking annotations for 24 videos from /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation
2152
Loading tracking annotations for 15 videos from /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation
1341
Loading tracking annotations for 16 videos from /kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_tracking_annotation/volleyball_tracking_annotation
File size: 0.52 MB


In [ ]:
import pickle
import random

import numpy as np
import cv2
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
# from src.paths import VIDEOS_DIR, PERSON_ANNOTATIONS_DIR

from PIL import Image

class VolleyBallPersonDataset(Dataset):

    def __init__(self, data_list, preprocess=None, shuffle=False):
        # super().__init__(self)
        self.data_list = data_list
        self.preprocess = preprocess
        self._shuffle(shuffle)

    # At first, we train on person activity so we will feed the boxes alone with no group activity

    def __getitem__(self, idx):
        """
        Returns a single sample: (processed_crops, labels)

        Returns:
            - processed_crops: tensor [12, 3, 224, 224] (length 12, padded if needed)
            - processed_labels: tensor [12] with -1 for padding
            :param idx:
            :return:
        """
        vid, clip, frame = self.data_list[idx]['video'], self.data_list[idx]['clip'], self.data_list[idx]['frame_id']

        image_path = VIDEOS_DIR / vid / clip / f'{frame}.jpg'
        image = Image.open(image_path).convert('RGB')

        boxes, player_category = self.data_list[idx]['boxes'], self.data_list[idx]['category']

        processed_crops = []
        processed_labels = []

        # ✅ FIXED: Process each box correctly
        for box, label in zip(boxes, player_category):
            # cropped_box = image.crop(box)
            # processed_crop = preprocessor(image.crop(box))
            # processed_crops.append(self.preprocess(image.crop(box)))
            processed_crops.append(self.preprocess(image.crop(box)))
            processed_labels.append(label)

        # Pad to exactly 12 players with zero tensors and -1 labels
        while len(processed_crops) < 12:
            zero_crop = torch.zeros((3, 224, 224), dtype=torch.float32)
            processed_crops.append(zero_crop)
            processed_labels.append(-1)

        # Ensure exactly 12 players (truncate if more)
        processed_crops = torch.stack(processed_crops[:12])
        processed_labels = torch.tensor(processed_labels[:12], dtype=torch.long)

        # ✅ FIXED: Close image to free memory (important with num_workers)
        image.close()

        return processed_crops, processed_labels


    def __len__(self):
        return len(self.data_list)

    def len_12_players(self):
        return len(self.data_list) * 12

    def _shuffle(self, shuffle):
        if shuffle:
            random.shuffle(self.data_list)


def collate_fn(batch):
    crops, labels = zip(*batch)
    images = torch.stack(crops)
    labels = torch.stack(labels)

    return images, labels


In [23]:
import torchvision.transforms as transforms

def person_preprocessor():
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomRotation(15),
        transforms.RandomGrayscale(p=0.1),
        transforms.RandomPerspective(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    return train_transform, val_transform


In [24]:
from IPython.display import FileLink
from IPython.display import display

# annot_dir = "volleyball-baseline-annotations"

annot_train_pkl = "/kaggle/working/training-outputs/train_players_annots.pickle"
annot_val_pkl = "/kaggle/working/training-outputs/val_players_annots.pickle"
# annot_test_pkl = "/kaggle/working/training-outputs/test_players_annots.pickle"


# path = f'{annot_dir}/b3-test-annot.pickle'
# if not os.path.exists(annot_dir):
#     os.makedirs(annot_dir)
   
# display(FileLink(annot_train_pkl, result_html_prefix="click here to download: "))
# display(FileLink(annot_val_pkl, result_html_prefix="click here to download: "))
# display(FileLink(annot_test_pkl, result_html_prefix="click here to download: "))

In [25]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, best_model_path: str = 'best_checkpoint.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.best_model_path = best_model_path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss

        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model)

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.best_model_path)

In [32]:
import numpy as np
import torch
from torch import nn
import torchvision.models as models

# Baseline 1 is working on the image level with spatial model. >> No temporal <<
# The model is based on fine-tuning pretrained resnet50 on fc7 layer
#
class PersonActivityClassifier(nn.Module):
    def __init__(self, num_classes):
        super(PersonActivityClassifier, self).__init__()
        self.num_classes = num_classes

        model = models.resnet50(pretrained=True)
        model = nn.Sequential(*(list(model.children())[:-1]))

        fc_layers = nn.Sequential(
            nn.Dropout(0.5, inplace=False),
            nn.Linear(2048, self.num_classes)
        )
        self.backbone_model = model
        self.classifier = fc_layers

        self.optimizer = None
        self.criterion = None
        self.accuracy = None
        self.save_interval = None
        self.early_stopping = None


    def model_summary(self):
        print(f'backbone model\n {self.backbone_model}')

        print(f'classifier')
        print(self.classifier)

    def _optimizers(self, optim):
        optims = dict(
            Adam=torch.optim.Adam([{'params': self.backbone_model.parameters()},
                                   {'params': self.classifier.parameters()}],
                                  lr=optim['lr'],
                                  weight_decay=optim['weight_decay']),
            AdamW=torch.optim.AdamW([{'params': self.backbone_model.parameters()},
                                   {'params': self.classifier.parameters()}],
                                  lr=optim['lr'],
                                  weight_decay=optim['weight_decay']),
            SGD=torch.optim.SGD([{'params': self.backbone_model.parameters()},
                                 {'params': self.classifier.parameters()}],
                                lr=optim['lr'],
                                weight_decay=optim['weight_decay'])
        )
        return optims[optim['optimizer']]

    def metrics(self, optimizer, criterion, accuracy, save_interval=5, early_stopping=None):
        self.optimizer = self._optimizers(optimizer)
        self.criterion = criterion
        self.accuracy = accuracy
        self.save_interval = save_interval
        self.early_stopping = early_stopping

    def forward(self, x):
        x = self.backbone_model(x)
        # output.shape == [B * 12, 2048, 1, 1]
        # print(f'B * 12{x.size(0)}')  # B * 12
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x




In [27]:
import yaml

class Config:
    def __init__(self, config_dict):
        self.paths = config_dict.get("paths", {})
        self.dataset = config_dict.get("dataset", {})
        self.model = config_dict.get("model", {})
        self.training = config_dict.get("training", {})
        self.experiment = config_dict.get("experiment", {})

    def __repr__(self):
        return f"Config(model={self.model}, training={self.training}, data={self.dataset}, experiment={self.experiment})"


def load_config(config_path="config.yaml"):
    with open(config_path, "r") as file:
        config = yaml.safe_load(file)
    config = Config(config)
    return config

In [33]:
def save_checkpoint(model, optimizer, epoch, val_loss, save_path):
    # Todo later
    checkpoint = {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch,
        'val_loss': val_loss
    }
    torch.save(checkpoint, save_path / 'checkpoint.pth')

In [38]:
from torch.utils.data import DataLoader

class DummyPlayerDataset(Dataset):
    def __init__(self, num_frames=100, num_players=12, num_classes=5):
        self.num_frames = num_frames
        self.num_players = num_players
        self.num_classes = num_classes

    def __len__(self):
        return self.num_frames

    def __getitem__(self, idx):

        # 12 player images in one frame
        players = torch.randn(
            self.num_players,
            3,
            224,
            224
        )

        # One label for each player
        labels = torch.randint(
            0,
            self.num_classes,
            (self.num_players,)
        )

        return players, labels


dataset = DummyPlayerDataset()

dummies_dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

players, labels = next(iter(dummies_dataloader))

print("Players:", players.shape)
print("Labels:", labels.shape)

Players: torch.Size([4, 12, 3, 224, 224])
Labels: torch.Size([4, 12])


In [42]:
def seq_12_players_length(data_len):
    data_len = data_len * 12
    assert data_len % 12 == 0
    return data_len

In [ ]:
import torch
import pickle

from torch.utils.data import DataLoader


def train_epoch(model, trainLoader, optimizer, criterion, device):
    model.train()

    running_loss = 0
    epoch_correct_predictions = 0

    for batch_idx, (data, target) in enumerate(trainLoader):
        # The input shape is x: [B, 12, 3, 224, 224]
        B, P, C, H, W = data.shape

        data = data.view(B * P, C, H, W)
        target = target.view(B * P)

        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        output = model(data)

        loss = criterion(output, target)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)

        prediction = torch.argmax(output, dim=1)
        batch_correct_predictions = (prediction == target).sum().item()

        epoch_correct_predictions += batch_correct_predictions

    epoch_loss = running_loss / trainLoader.dataset.len_12_players()
    epoch_accuracy = epoch_correct_predictions / trainLoader.dataset.len_12_players()

    return model, optimizer, epoch_loss, epoch_accuracy


def eval_model(model, valLoader, criterion, device):
    model.eval()
    running_loss = 0
    epoch_correct_predictions = 0

    with torch.no_grad():

        for batch_idx, (data, target) in enumerate(valLoader):
            B, P, C, H, W = data.shape
            data = data.view(B * P, C, H, W)
            target = target.view(B * P)

            data, target = data.to(device), target.to(device)
            output = model(data)

            loss = criterion(output, target)
            running_loss += loss.item() * data.size(0)

            prediction = torch.argmax(output, dim=1)
            batch_correct_predictions = (prediction == target).sum().item()

            epoch_correct_predictions += batch_correct_predictions

    epoch_loss = running_loss / valLoader.dataset.len_12_players()
    epoch_accuracy = epoch_correct_predictions / valLoader.dataset.len_12_players()

    return epoch_loss, epoch_accuracy


def fit(model, trainLoader, valLoader, epochs, optimizer, criterion, output_dir, device):
    print(f"Training on device: {device}")
    model.to(device)
    early_stopping = EarlyStopping(patience=5, min_delta=0.001)

    train_dataset_length = len(trainLoader.dataset)
    num_of_steps = 0

    optimizer = model.optimizer

    for epoch in range(epochs):
        num_of_steps += train_dataset_length
        print(f'epoch: {epoch + 1}/{epochs}, steps: {num_of_steps}/{train_dataset_length * epochs}')
        model, optimizer, train_loss, train_accuracy = train_epoch(model,
                                                                   trainLoader,
                                                                   optimizer,
                                                                   criterion,
                                                                   device)

        val_loss, val_accuracy = eval_model(model, valLoader, criterion, device)

        print(f'\ttrain loss: {train_loss:.4f} - train accuracy: {(train_accuracy * 100):.2f}%,'
              f' val loss: {val_loss: .4f} - val accuracy: {(val_accuracy * 100): .2f} % ')
        # todo
        if epoch % model.save_interval == 0:
            save_checkpoint(model, optimizer, epoch, val_loss, output_dir)

        # todo
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break


def train_model(model_configs, train_configs, output_checkpoint_dir):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_data_path = PERSON_ANNOTATIONS_DIR / 'train_players_crops.pickle'
    val_data_path = PERSON_ANNOTATIONS_DIR / 'val_players_crops.pickle'

    train_preprocess, val_preprocessor = person_preprocessor()

    with open(train_data_path, 'rb') as tr, open(val_data_path, 'rb') as vl:
        train_data = pickle.load(tr)
        val_data = pickle.load(vl)

    dataset = VolleyBallPersonDataset(train_data, preprocess=train_preprocess)

    print(f'dataset size: {len(dataset)}')

    batch_size = train_configs['batch_size']
    num_workers = train_configs['num_workers']

    train_loader = DataLoader(VolleyBallPersonDataset(train_data, preprocess=train_preprocess),
                              batch_size=batch_size, num_workers=num_workers)
    val_loader = DataLoader(VolleyBallPersonDataset(val_data, preprocess=val_preprocessor),
                            batch_size=batch_size, num_workers=num_workers)

    num_classes = model_configs['num_classes']
    model = PersonActivityClassifier(num_classes)

    optimizer = {
        "optimizer": train_configs['optimizer'],
        "lr": train_configs['learning_rate'],
        "weight_decay": train_configs['weight_decay']
    }

    criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)
    accuracy = "accuracy"

    save_interval = 5
    if train_configs['early_stopping']:
        early_stopping = EarlyStopping()
    else:
        early_stopping = None

    model.metrics(optimizer=optimizer,
                  criterion=criterion,
                  accuracy=accuracy,
                  save_interval=save_interval,
                  early_stopping=early_stopping)

    epochs = train_configs['epochs']

    # model.model_summary()
    # fit(model, dummies_dataloader, dummies_dataloader, epochs, optimizer, criterion, output_dir=output_checkpoint_dir, device=device)

    fit(model, train_loader, val_loader, epochs, optimizer, criterion, output_dir=output_checkpoint_dir, device=device)



In [44]:
'/kaggle/input/datasets/mahmoudkhaled923/configs/b3_person_train_config.yaml'
b3_config_path = Path('/kaggle') / 'input' / 'datasets' / 'mahmoudkhaled923' / 'configs' / 'b3_person_train_config.yaml'
b3_config = load_config(b3_config_path)
b3_person_checkpoint_dir = OUTPUT_DIR / 'b3' / 'person_activity'
if not os.path.exists(b3_person_checkpoint_dir):
        os.makedirs(b3_person_checkpoint_dir)


model_configs = b3_config.model
train_configs = b3_config.training
# print(train_configs)

In [47]:
train_model(model_configs, train_configs, b3_person_checkpoint_dir)

dataset size: 2152
Training on device: cuda
epoch: 1/25, steps: 2152/53800
	train loss: 1.0095 - train accuracy: 69.77%, val loss:  1.0313 - val accuracy:  70.36 % 
epoch: 2/25, steps: 4304/53800
	train loss: 0.8820 - train accuracy: 72.30%, val loss:  0.9338 - val accuracy:  71.33 % 
epoch: 3/25, steps: 6456/53800
	train loss: 0.8436 - train accuracy: 72.91%, val loss:  1.0676 - val accuracy:  64.66 % 
epoch: 4/25, steps: 8608/53800
	train loss: 0.8070 - train accuracy: 73.69%, val loss:  0.9666 - val accuracy:  68.41 % 
epoch: 5/25, steps: 10760/53800
	train loss: 0.7899 - train accuracy: 74.38%, val loss:  1.3673 - val accuracy:  60.65 % 
epoch: 6/25, steps: 12912/53800
	train loss: 0.7709 - train accuracy: 74.71%, val loss:  1.0813 - val accuracy:  65.75 % 
epoch: 7/25, steps: 15064/53800
	train loss: 0.7528 - train accuracy: 75.16%, val loss:  0.9107 - val accuracy:  71.35 % 
epoch: 8/25, steps: 17216/53800
	train loss: 0.7328 - train accuracy: 75.88%, val loss:  1.4779 - val accu

: 

after it took 220 minutes for one epoch, i decided to stop the training and try to check the problem

### Woooooooooooooooooooooow 
49 min for one epoch
My model is Woooooooooorking

epoch: 1/25, steps: 2152/53800
	train loss: 0.8390 - train accuracy: 73.51%, val loss: 0.7836 - val accuracy: 74.69%
epoch: 2/25, steps: 4304/53800
	train loss: 0.6255 - train accuracy: 78.81%, val loss: 0.7476 - val accuracy: 75.37%
epoch: 3/25, steps: 6456/53800
	train loss: 0.5695 - train accuracy: 80.46%, val loss: 0.7664 - val accuracy: 75.28%
epoch: 4/25, steps: 8608/53800
	train loss: 0.5372 - train accuracy: 81.48%, val loss: 0.7917 - val accuracy: 76.07%
epoch: 5/25, steps: 10760/53800
	train loss: 0.5179 - train accuracy: 81.89%, val loss: 0.7266 - val accuracy: 76.58%
Checkpoint saved at epoch 5
epoch: 6/25, steps: 12912/53800
	train loss: 0.4989 - train accuracy: 82.63%, val loss: 0.7358 - val accuracy: 76.96%
epoch: 7/25, steps: 15064/53800
	train loss: 0.4879 - train accuracy: 82.86%, val loss: 0.7718 - val accuracy: 75.62%
epoch: 8/25, steps: 17216/53800
	train loss: 0.4819 - train accuracy: 82.85%, val loss: 0.7732 - val accuracy: 75.90%
epoch: 9/25, steps: 19368/53800


### i started the model training in 10:00 PM
### it Taked 30 minutes to train the first baseline for 48 epochs 
#### there was a mistake in the above try "i forgot a condition to stop the eval on step 3"

### Started second training try in 11:00 PM for 100 epochs
### It Taked about 2 HOURS 


### I Started the third try in tuesday at 7:00 PM 

### I started for 80 epochs in 10:25 AM
### Ended in ........

In [ ]:
ty = "/a/b/c/d/f"
print(ty.replace("/a/b/", ""))

In [ ]:
# backbone_model_state_path = training_output_path+'/backbone_model_state_dict.pth'
# classifier_model_state_path = training_output_path+'/classifier_state_dict.pth'

# test_model = ImageLevelModel(num_classes)
# with open(backbone_model_state_path, 'rb') as backnone, open(classifier_model_state_path, 'rb') as classifier:
#     test_model.backbone_model.load_state_dict(torch.load(backnone))
#     test_model.classifier.load_state_dict(torch.load(classifier))

# test_path = root_videos_dataset
# test_loader = DataLoader(VolleyBallDataSet(test_path, train_annot, preprocess=preprocess), batch_size=batch_size)

# test_acc = test_model.test_model(test_loader)
# print(test_acc)